# Fusion et publication du modèle

Le modèle servi par vLLM (`dpo-v1`) est le modèle de base dans lequel on a ajouté deux adaptateurs LoRA : celui du SFT (notebook 04), puis celui du DPO (notebook 05). Un adaptateur ne contient que les petites tables entraînées par LoRA, environ 70 Mo. La fusion (`merge_and_unload`) ajoute leurs nombres aux poids du modèle : on obtient un modèle ordinaire, de la même taille que le modèle de base, que vLLM charge directement.

Ce notebook refait la fusion à partir des adaptateurs enregistrés, vérifie le résultat sur un cas, puis publie le modèle sur Hugging Face avec les deux adaptateurs et une fiche.

In [1]:
import gc
import json
import sys
from pathlib import Path

import torch
from huggingface_hub import HfApi
from peft import PeftModel
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.append("../api")
from app.llm import MAX_TOKENS, STOP_TOKEN_IDS, build_prompt  # le prompt exact de l'API

MODEL = "Qwen/Qwen3-1.7B-Base"
SFT_DIR = Path("../models/sft/final")
DPO_DIR = Path("../models/dpo/final")
VERSION = "dpo-v1"
SERVE_DIR = Path("../models/serve") / VERSION
MODEL_REPO = "g-hostin/medical-triage-poc-model"

for folder in (SFT_DIR, DPO_DIR):
    info = json.loads((folder / "run_info.json").read_text(encoding="utf-8"))
    print(folder.parent.name, ":", info["name"], "| checkpoint", info["best_checkpoint"], "| données", info["dataset_revision"])

C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sft : run1_lr2e-4 | checkpoint checkpoint-1200 | données v1.1
dpo : run_lr5e-5 | checkpoint checkpoint-250 | données v1.1


## 1. La fusion

Même ordre qu'à l'entraînement : le DPO a été entraîné sur le modèle SFT déjà fusionné (notebook 05). On fusionne donc l'adaptateur du SFT, puis celui du DPO. Tout reste en bf16, le format des poids du modèle de base.

In [2]:
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)
n_base = sum(p.numel() for p in base.parameters())

sft = PeftModel.from_pretrained(base, SFT_DIR).merge_and_unload()
merged = PeftModel.from_pretrained(sft, DPO_DIR).merge_and_unload()
n_merged = sum(p.numel() for p in merged.parameters())
print(f"paramètres avant la fusion : {n_base:,} | après : {n_merged:,}".replace(",", " "))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   5%|▍         | 15/310 [00:00<00:02, 139.26it/s]

Loading weights:   9%|▉         | 29/310 [00:00<00:03, 70.33it/s] 

Loading weights:  12%|█▏        | 38/310 [00:00<00:04, 61.44it/s]

Loading weights:  15%|█▍        | 46/310 [00:00<00:04, 64.05it/s]

Loading weights:  19%|█▊        | 58/310 [00:00<00:03, 77.86it/s]

Loading weights:  22%|██▏       | 67/310 [00:00<00:03, 63.99it/s]

Loading weights:  46%|████▋     | 144/310 [00:01<00:00, 227.30it/s]

Loading weights: 100%|██████████| 310/310 [00:01<00:00, 286.39it/s]

paramètres avant la fusion : 1 720 574 976 | après : 1 720 574 976


Le nombre de paramètres ne change pas : les tables LoRA ont été ajoutées aux poids existants, pas à côté.

## 2. Comparaison avec le modèle servi

Sur notre PC, `models/serve/dpo-v1` existe déjà : c'est le dossier que vLLM sert depuis le 23 novembre, fusionné à l'époque par un petit script. Plutôt que de l'écraser, on compare les poids. Sur une autre machine, le dossier n'existe pas et la cellule l'enregistre.

In [3]:
weights = merged.state_dict()
if SERVE_DIR.exists():
    served = load_file(SERVE_DIR / "model.safetensors")
    common = sorted(set(weights) & set(served))
    gap = max((weights[k].float() - served[k].float()).abs().max().item() for k in common)
    print("tenseurs :", len(weights), "après la fusion,", len(served), "dans le dossier servi,", len(common), "en commun")
    print("absents du dossier servi :", sorted(set(weights) - set(served)))
    print("plus grand écart entre deux poids :", gap)
    del served
else:
    merged.save_pretrained(SERVE_DIR)
    AutoTokenizer.from_pretrained(SFT_DIR).save_pretrained(SERVE_DIR)
    print("modèle enregistré dans", SERVE_DIR)
print(sorted(p.name for p in SERVE_DIR.iterdir()))

del base, sft, merged, weights
_ = gc.collect()

tenseurs : 311 après la fusion, 310 dans le dossier servi, 310 en commun
absents du dossier servi : ['lm_head.weight']
plus grand écart entre deux poids : 0.0


['chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


## 3. Vérification sur un cas

On charge le dossier servi, comme vLLM, et on lui donne le cas du bouton « douleur thoracique » de la page de démo, avec le prompt de l'API (`api/app/llm.py`) et une génération sans hasard.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(SERVE_DIR)
model = AutoModelForCausalLM.from_pretrained(SERVE_DIR, dtype=torch.bfloat16, device_map="cuda")

case = "Homme de 55 ans, douleur thoracique en étau depuis 40 minutes, qui irradie dans le bras gauche, avec des sueurs."
inputs = tokenizer(build_prompt(case), return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=MAX_TOKENS, do_sample=False, eos_token_id=STOP_TOKEN_IDS)
answer = output[0][inputs.input_ids.shape[1]:]
print(tokenizer.decode(answer, skip_special_tokens=True))
print()
print("tokens générés :", len(answer), "| arrêt seul :", answer[-1].item() in STOP_TOKEN_IDS)

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 21/310 [00:00<00:01, 193.55it/s]

Loading weights:  14%|█▍        | 43/310 [00:00<00:01, 177.62it/s]

Loading weights:  21%|██▏       | 66/310 [00:00<00:01, 194.50it/s]

Loading weights:  28%|██▊       | 88/310 [00:00<00:01, 200.27it/s]

Loading weights:  36%|███▌      | 111/310 [00:00<00:00, 200.07it/s]

... (6 outputs left out)

Loading weights:  85%|████████▍ | 263/310 [00:01<00:00, 202.24it/s]

Loading weights:  92%|█████████▏| 285/310 [00:01<00:00, 207.20it/s]

Loading weights: 100%|██████████| 310/310 [00:01<00:00, 205.22it/s]

Niveau : urgence maximale (tri 2 de l'échelle FRENCH).
Explication : Homme de 55 ans présentant une douleur thoracique en étau depuis 40 minutes, avec irradiation du bras gauche et des sueurs, ce qui correspond au critère « douleur thoracique » du motif « Douleur thoracique ». Le délai dépasse 30 minutes, donc le tri par défaut 2 s'applique. Le caractère brut (sueur) et l'irradiation du bras gauche font craindre une ischémie cardiaque, ce qui justifie une prise en charge rapide.
Conduite : Installer le patient en box, mesurer rapidement les constantes (PA, FC, SpO2, température, ECG), prévenir le médecin pour une évaluation dans les 20 minutes et prévoir une bandelette urinaire et un ECG en urgence.

tokens générés : 204 | arrêt seul : True


## 4. Publication sur Hugging Face

La fiche du modèle est dans le dépôt (`model/README.md`) : contenu, données, réglages, résultats des notebooks 06 et 07 et limites d'usage. Le modèle est publié en public, comme le jeu de données. Le dossier servi va à la racine, les deux adaptateurs dans `adapters/` avec leur `run_info.json`, et l'étiquette `dpo-v1` fixe la version, comme `v1.1` pour les données. Il faut être connecté (`uv run hf auth login`).

In [5]:
api = HfApi()
api.create_repo(MODEL_REPO, repo_type="model", private=False, exist_ok=True)
api.upload_folder(repo_id=MODEL_REPO, folder_path=SERVE_DIR, commit_message=f"Modèle fusionné {VERSION}")
for name, folder in (("sft", SFT_DIR), ("dpo", DPO_DIR)):
    api.upload_folder(repo_id=MODEL_REPO, folder_path=folder, path_in_repo=f"adapters/{name}",
                      allow_patterns=["adapter_config.json", "adapter_model.safetensors", "run_info.json"],
                      commit_message=f"Adaptateur LoRA du {name.upper()}")
api.upload_file(repo_id=MODEL_REPO, path_or_fileobj="../model/README.md", path_in_repo="README.md",
                commit_message="Fiche du modèle")
api.create_tag(MODEL_REPO, tag=VERSION, exist_ok=True)

info = api.model_info(MODEL_REPO, files_metadata=True)
print("public :", not info.private)
for f in info.siblings:
    print(f"  {f.rfilename} ({f.size / 1e6:,.1f} Mo)".replace(",", " "))

No files have been modified since last commit. Skipping to prevent empty commit.


No files have been modified since last commit. Skipping to prevent empty commit.


No files have been modified since last commit. Skipping to prevent empty commit.


No files have been modified since last commit. Skipping to prevent empty commit.


public : True
  .gitattributes (0.0 Mo)
  README.md (0.0 Mo)
  adapters/dpo/adapter_config.json (0.0 Mo)
  adapters/dpo/adapter_model.safetensors (69.8 Mo)
  adapters/dpo/run_info.json (0.0 Mo)
  adapters/sft/adapter_config.json (0.0 Mo)
  adapters/sft/adapter_model.safetensors (69.8 Mo)
  adapters/sft/run_info.json (0.0 Mo)
  chat_template.jinja (0.0 Mo)
  config.json (0.0 Mo)
  generation_config.json (0.0 Mo)
  model.safetensors (3 441.2 Mo)
  tokenizer.json (11.4 Mo)
  tokenizer_config.json (0.0 Mo)


## Bilan

- la fusion redonne exactement le modèle servi depuis le 23 novembre : sur les 310 tenseurs du dossier, le plus grand écart est 0,0. Le tenseur en plus, `lm_head.weight`, n'est pas enregistré à part, car Qwen3-1.7B réutilise la table des mots pour la couche de sortie (`tie_word_embeddings`)
- le nombre de paramètres ne change pas avec la fusion : 1 720 574 976 avant et après
- sur le cas de la douleur thoracique, le modèle répond « urgence maximale (tri 2) » et s'arrête seul après 204 tokens. L'explication a des tournures bizarres (« le caractère brut (sueur) ») et la conduite cite une bandelette urinaire : le niveau est bon, la relecture par un soignant reste nécessaire
- le modèle est public sur Hugging Face avec l'étiquette `dpo-v1` : le modèle fusionné (3 441,2 Mo), les deux adaptateurs (69,8 Mo chacun) avec leur `run_info.json`, et la fiche
- ce passage est le second : les fichiers n'avaient pas changé, donc Hugging Face n'a pas créé de nouveau commit (« Skipping to prevent empty commit »)

Pour une nouvelle version du modèle, on change `VERSION` et les dossiers des adaptateurs, puis on relance ce notebook. L'ancienne version reste disponible sous son étiquette.